In [ ]:
from glob import glob
import os
import pandas as pd
import json
from src.utils.preprocessing import Preprocessor

def check_label(df, s):
    df = df.to_frame()
    df['label'] = s
    return df


# date-stamped folder removed; using neutral role paths

In [ ]:
agg_df_save_dir = os.path.join('data','reviewed_labels')
load_path = sorted([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i])


In [ ]:
load_path

In [ ]:
merged_all = pd.read_excel([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i][1])


In [ ]:

cancer_dict_path = os.path.join('.', 'cancer_dict.json')
with open(cancer_dict_path, 'r', encoding='utf-8') as file:
    cancer_dict = json.load(file)
cancer_en_dict = {int(k) : v['name_en'] for k, v in cancer_dict.items()}
cancer_order = list(cancer_en_dict.values()) + ['Missing']

cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Merged'
cnacer_merged_en_dict = {k:v if v not in cancer_merge_keys else merged_cancer_label for k, v in cancer_en_dict.items() }

In [ ]:
def age_group(age):
    for n, (sty, edy) in enumerate(zip(age_bins[:-1], age_bins[1:])):
        if sty <= age < edy:
            return n
    else:
        return n + 1

age_bins = [0] + list(range(40, 90, 10))
print(age_bins)

age_dict = {}
for n, (st, ed) in enumerate(zip(age_bins, age_bins[1:])):
    strs = r'~'
    if n > 0:
        strs = f'{st}{strs}'
    strs = f'{strs}{ed}'
    age_dict[n] = strs
else:
    age_dict[n+1] = f'{ed}~'
print(age_dict)


    # return len(age_bins)


In [ ]:
# Metastasis와 Recurrence 컬럼을 one-hot encoding
metastasis_ohe = pd.get_dummies(merged_all['Metastasis'], prefix='Metastasis')
recurrence_ohe = pd.get_dummies(merged_all['Recurrence'], prefix='Recurrence')

# merged_all에 one-hot encoding된 컬럼들 추가
merged_all = pd.concat([merged_all, metastasis_ohe, recurrence_ohe], axis=1)

In [ ]:
from itertools import product

target_names = ['Recurrence','Metastasis']
target_classes = ['negative','uncertain','positive']
target_combs = ['_'.join(i) for i in product(target_names,target_classes)]
# target_combs

In [ ]:
# 병원등록번호와 수술일자를 합쳐서 새로운 컬럼 '병원등록번호_수술일자' 생성
merged_all['rmc'] = merged_all['병원등록번호'].astype(str) + '_' + merged_all['수술일자'].astype(str) +'_' + merged_all.index.astype(str) 


In [ ]:
merged_all['검사나이구간'] = merged_all['검사나이'].apply(age_group)
column_dfs = []

for row in ['성별','검사나이구간','암종']:
    temp_dfs = []
    temp_numb_dfs = []
    for col in ['병원등록번호','rmc']:
        temp_numb = merged_all.groupby(row)[col].nunique()
        str_temp_numb = temp_numb.apply(lambda x: f"{x:,}") + ' (' + (temp_numb / temp_numb.sum() * 100).round(1).astype(str) + '%)'
        if row == '검사나이구간':
            str_temp_numb = str_temp_numb.sort_index()
            str_temp_numb.index = [age_dict[i] for i in str_temp_numb.index]
        elif row == '암종':
            str_temp_numb = str_temp_numb.reindex(cancer_order)
        print(temp_numb.sum())
        
        temp_dfs.append(str_temp_numb)
        temp_numb_dfs.append(temp_numb)
    temp_dfs = pd.concat(temp_dfs, axis = 1)
    temp_numb_dfs = pd.concat(temp_numb_dfs, axis = 1)
    temp_dfs.loc['Sum'] = temp_numb_dfs.sum().apply(lambda x: f"{x:,}") + ' (100.0%)'
    
    # print(temp_dfs.sum())
    column_dfs.append(temp_dfs)
column_dfs = pd.concat(column_dfs, axis = 0)

    # sex_numb = merged_all.groupby('성별')[col].nunique() # 성별 별 등록번호 
    # year_numb = merged_all.groupby('검사나이구간')[col].nunique().rename(index = age_dict) # 성별 별 등록번호 
    # cancer_numb = merged_all.groupby('암번호')[col].nunique().rename(index = cancer_en_dict)

    # str_sex_numb = sex_numb.astype(str) + ' (' + (sex_numb / sex_numb.sum() * 100).round(1).astype(str) + '%)'
    # str_year_numb = year_numb.astype(str) + ' (' + (year_numb / year_numb.sum() * 100).round(1).astype(str) + '%)'
    # str_cancer_numb = cancer_numb.astype(str) + ' (' + (cancer_numb / cancer_numb.sum() * 100).round(1).astype(str) + '%)'

    # subs = pd.concat([str_sex_numb,str_year_numb,str_cancer_numb])

    # column_dfs.append(subs)

In [ ]:
merged_all['검사나이구간'] = merged_all['검사나이'].apply(age_group)
column_dfs = []

for row in ['성별','검사나이구간','암종']:
    temp_dfs = []
    temp_numb_dfs = []
    for col in ['병원등록번호','rmc']:
        temp_numb = merged_all.groupby(row)[col].nunique()
        str_temp_numb = temp_numb.apply(lambda x: f"{x:,}") + ' (' + (temp_numb / temp_numb.sum() * 100).round(1).astype(str) + '%)'
        if row == '검사나이구간':
            str_temp_numb = str_temp_numb.sort_index()
            str_temp_numb.index = [age_dict[i] for i in str_temp_numb.index]
        elif row == '암종':
            str_temp_numb = str_temp_numb.reindex(cancer_order)
        print(temp_numb.sum())
        
        temp_dfs.append(str_temp_numb)
        temp_numb_dfs.append(temp_numb)
    temp_dfs = pd.concat(temp_dfs, axis = 1)
    temp_numb_dfs = pd.concat(temp_numb_dfs, axis = 1)
    temp_dfs.loc['Sum'] = temp_numb_dfs.sum().apply(lambda x: f"{x:,}") + ' (100.0%)'
    
    # print(temp_dfs.sum())
    column_dfs.append(temp_dfs)
column_dfs = pd.concat(column_dfs, axis = 0)

    # sex_numb = merged_all.groupby('성별')[col].nunique() # 성별 별 등록번호 
    # year_numb = merged_all.groupby('검사나이구간')[col].nunique().rename(index = age_dict) # 성별 별 등록번호 
    # cancer_numb = merged_all.groupby('암번호')[col].nunique().rename(index = cancer_en_dict)

    # str_sex_numb = sex_numb.astype(str) + ' (' + (sex_numb / sex_numb.sum() * 100).round(1).astype(str) + '%)'
    # str_year_numb = year_numb.astype(str) + ' (' + (year_numb / year_numb.sum() * 100).round(1).astype(str) + '%)'
    # str_cancer_numb = cancer_numb.astype(str) + ' (' + (cancer_numb / cancer_numb.sum() * 100).round(1).astype(str) + '%)'

    # subs = pd.concat([str_sex_numb,str_year_numb,str_cancer_numb])

    # column_dfs.append(subs)

In [ ]:
column_dfs

In [ ]:
full_dfs = []
for row in ['성별','검사나이구간','암종']:
    # temp_dfs = []
    # temp_numb_dfs = []
    # 각 col1(Recurrence, Metastasis) 기준으로
    category_dfs = []
    for col1 in target_names:
        # col1에 대한 그룹별 전체 값 (분모)
        target_temp_dfs = []
        str_target_temp_dfs = []
        total_numb = merged_all.groupby(row)[[col1 + '_' + cls for cls in target_classes]].sum().sum(axis=1)
        
        

        for col2 in target_classes:
            temp_numb = merged_all.groupby(row)[col1 + '_' + col2].sum()
            # 행별(각 row 그룹별)로, col1의 세부값(col2)에 해당하는 수가 col1전체에서 몇 %를 차지하는지
            # percent = (temp_numb / total_numb * 100).round(1)
            # str_temp_numb = temp_numb.apply(lambda x: f"{x:,}") + ' (' + percent.astype(str) + '%)'
            # if row == '검사나이구간':
            #     str_temp_numb = str_temp_numb.rename(index = age_dict)
            # elif row == '암번호':
            #     str_temp_numb = str_temp_numb.rename(index = cancer_en_dict)
            
            # str_target_temp_dfs.append(str_temp_numb)
            target_temp_dfs.append(temp_numb)
            
        target_temp_dfs = pd.concat(target_temp_dfs, axis = 1)
        target_temp_dfs.loc['Sum'] = target_temp_dfs.sum()
        percent = (target_temp_dfs / target_temp_dfs.sum(1).values.reshape(-1, 1) * 100).round(1)

        str_temp_numb = target_temp_dfs.map(lambda x: f"{x:,}") + ' (' + percent.astype(str) + '%)'
        if row == '검사나이구간':
            # str_temp_numb = str_temp_numb.sort_index()
            str_temp_numb.index = [age_dict[i] if i in age_dict else i for i in str_temp_numb.index]
        elif row == '암종':
            str_temp_numb = str_temp_numb.reindex(cancer_order + ['Sum'])

        # str_temp_numb = target_temp_dfs.map(lambda x: f"{x:,}") + ' (' + percent.astype(str) + '%)'
        # if row == '검사나이구간':
        #     str_temp_numb = str_temp_numb.rename(index = age_dict)
        # elif row == '암종':
        #     str_temp_numb = str_temp_numb.rename(index = cancer_en_dict)


        # target_temp_dfs = target_temp_dfs
        
        # str_target_temp_dfs = pd.concat(str_target_temp_dfs, axis = 1)
        # str_target_temp_dfs.columns = target_temp_dfs.columns[:str_target_temp_dfs.shape[1]]
        
        category_dfs.append(str_temp_numb)
    category_dfs = pd.concat(category_dfs, axis = 1)
    full_dfs.append(category_dfs)
full_dfs = pd.concat(full_dfs, axis = 0)

In [ ]:
full_dfs.set_axis(pd.MultiIndex.from_product([target_names, [j.capitalize() for j in target_classes]]), axis = 1)

In [ ]:
MI = pd.MultiIndex.from_tuples(
    [('Sex', v) for v in ['F', 'M'] + ['Sum']] +
    [('Age', v) for v in list(age_dict.values()) + ['Sum']] +
    [('Cancer', v) for v in list(cancer_en_dict.values()) + ['Missing', 'Sum']],
    names=['Category', 'Value']
)
save_df = pd.concat([
    column_dfs.set_axis(pd.MultiIndex.from_product([['Information'], ['Patients','Reports']]), axis = 1), 
    full_dfs.set_axis(pd.MultiIndex.from_product([target_names, [j[:3].capitalize()+'.' for j in target_classes]]), axis = 1)
], axis = 1).set_axis(MI, axis = 0)

save_df.to_excel('Table description v03.xlsx')
save_df

In [ ]:
## 환자 수
sex_numb = merged_all.groupby('성별')['검사결과결론내용'].nunique() # 성별 별 등록번호 
year_numb = merged_all.groupby('검사나이구간')['검사결과결론내용'].nunique().rename(index = age_dict) # 성별 별 등록번호 
cancer_numb = merged_all.groupby('암번호')['검사결과결론내용'].nunique().rename(index = cancer_en_dict)

second_columns = pd.concat([
    sex_numb,
    year_numb,
    cancer_numb
])

In [ ]:
pd.concat([first_columns, second_columns], axis = 1)

In [ ]:
cancer_numb

In [ ]:
# 병원등록번호별로 성별, 검사나이, 암번호의 unique count를 구함
unique_counts = merged_all.groupby('병원등록번호')[['성별', '검사나이', '암번호']].nunique()
# print(unique_counts)
# set(unique_counts.values.reshape(-1))

In [ ]:


'data/reviewed_labels/all'

In [ ]:
dir_path = sorted(glob(os.path.join('data','reviewer','*','files','*.xlsx')))
meta1, meta2, recur = dir_path
# recur_df = pd.read_excel()

In [ ]:
meta1 = pd.read_excel(meta1)
meta2 = pd.read_excel(meta2)
recur = pd.read_excel(recur)


In [ ]:
## Train data
data_dir_path = sorted(glob(os.path.join('data','reviewed_labels','all', f'*train*')))
meta_train_df = pd.read_excel(data_dir_path[0])
recur_train_df = pd.read_excel(data_dir_path[1])

In [ ]:
import random


from pathlib import Path

def save_postprocessed_excel(df, original_path):
    # 원하는 label 순서 (탭 순서)
    label_order = ['positive', 'uncertain', 'negative']
    writer_path = Path(original_path).with_name('postprocessed-for-review.xlsx')
    df = df[['index','raw_text','prep_text','rule_label','prediction']]
    with pd.ExcelWriter(writer_path, engine='xlsxwriter') as writer:
        # all 시트: 전체 다
        df.to_excel(writer, index=False, sheet_name='all')
        # 각 label 별 시트 작성
        for label in label_order:
            df_label = df[df['prediction'] == label]
            df_label.to_excel(writer, index=False, sheet_name=label)
    print(f"Saved: {writer_path}")



def shuffle_list(list_data, random_seed=42):
    random_gen = random.Random(random_seed)  # random.Random 인스턴스 사용
    shuffled_list = list_data.copy()
    for _ in range(5):
        random_gen.shuffle(shuffled_list)
    return shuffled_list

# 예시 리스트
# example_list = list(range(10))  # 임의의 리스트
metas_index = shuffle_list(list(meta1.index))
recur_index = shuffle_list(list(recur.index))

# meta1
save_postprocessed_excel(meta1.iloc[metas_index], meta1._file_path if hasattr(meta1, '_file_path') else dir_path[0])
# meta2
save_postprocessed_excel(meta2.iloc[metas_index], meta2._file_path if hasattr(meta2, '_file_path') else dir_path[1])
# recur
save_postprocessed_excel(recur.iloc[recur_index], recur._file_path if hasattr(recur, '_file_path') else dir_path[2])


In [ ]:
raw_data_dir = glob(os.path.join('data','reports','*'))
print(raw_data_dir)
df = pd.read_csv(os.path.join(raw_data_dir[0]))


negative_patterns = ['without','not? (sugges|compat)', 'unlike', 'neither', 'no', 'negative']
uncertain_patterns = ['rule out', 'Rule Out', "Rule out"]

text_df = df['검사결과결론내용'].dropna()

preprocessor = Preprocessor(
    df = text_df,
    negative_patterns = negative_patterns,
    uncertain_patterns = uncertain_patterns
)

In [ ]:
df.head()

In [ ]:
df.groupby(['병원등록번호', '수술일자']).count()

In [ ]:
df.검사코드.value_counts().tolist()

In [ ]:
meta_df = pd.concat([check_label(d, k) for k, d in preprocessor.target_filtering('metas').items()]).sort_index()
recur_df = pd.concat([check_label(d, k) for k, d in preprocessor.target_filtering('recur').items()]).sort_index()



In [ ]:
def rename_prediction_to_label(df):
    """
    DataFrame의 'prediction' 컬럼 이름을 'label'로 변경합니다.
    해당 컬럼이 존재하지 않으면 아무런 동작도 하지 않습니다.
    """
    if 'prediction' in df.columns:
        return df.rename(columns={'prediction': 'label'})
    return df


def merge_dfs(df1, df2, df3):
    df2 = rename_prediction_to_label(df2)
    df23 = pd.concat([df2, df3.drop(columns = 'R_text')])
    return pd.merge(
        df1.drop(columns = ['label']).set_axis(['prep_text'], axis = 1), 
        df23[['prep_text','rule_label','label']],
        on = 'prep_text',
        how = 'left'
    )

meta_merged = merge_dfs(meta_df, meta2, meta_train_df).fillna('negative')
recur_merged = merge_dfs(recur_df, recur, recur_train_df).fillna('negative')

# meta2_renamed = rename_prediction_to_label(meta2)
# meta_label_df = pd.concat([meta2_renamed, meta_train_df.drop(columns = 'R_text')])
# meta_merged = pd.merge(
#     meta_df.drop(columns = ['label']).set_axis(['prep_text'], axis = 1), 
#     meta_label_df[['prep_text','rule_label','label']],
#     on = 'prep_text',
#     how = 'left'
# )

In [ ]:
df.loc[meta_merged.index, 'Metastasis'] = meta_merged.label
df.loc[recur_merged.index, 'Recurrence'] = recur_merged.label
for col in ['Metastasis', 'Recurrence']:
    df.loc[:,col] = df.loc[:,col].apply(lambda x: 'negative' if x == '' else x)

In [ ]:
agg_df_save_dir = os.path.join('data','reviewed_labels')
print(glob(os.path.join(agg_df_save_dir, '*')))
df.to_excel(os.path.join(agg_df_save_dir, 'all.xlsx'), index=False)
